In [1]:
!pip install transformers datasets peft accelerate bitsandbytes evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile

with zipfile.ZipFile("/content/drive/MyDrive/csci316 project/train_dataset.zip", "r") as z:
    z.extractall("/content/train_dataset")

with zipfile.ZipFile("/content/drive/MyDrive/csci316 project/val_dataset.zip", "r") as z:
    z.extractall("/content/val_dataset")

with zipfile.ZipFile("/content/drive/MyDrive/csci316 project/test_dataset.zip", "r") as z:
    z.extractall("/content/test_dataset")

In [4]:
from datasets import load_from_disk

train_tok = load_from_disk("/content/train_dataset")
val_tok   = load_from_disk("/content/val_dataset")
test_tok  = load_from_disk("/content/test_dataset")

print("Train:", train_tok)
print("Val:  ", val_tok)
print("Test: ", test_tok)

Train: Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 2464
})
Val:   Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 818
})
Test:  Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 818
})


In [5]:
train_tok.set_format("torch")
val_tok.set_format("torch")
test_tok.set_format("torch")

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=3
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    task_type=TaskType.SEQ_CLS,
    target_modules=["query", "value"],
    modules_to_save=["classifier"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 887,811 || all params: 278,933,766 || trainable%: 0.3183


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

epoch_logs = []

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [9]:
import torch
import numpy as np
from torch import nn
from transformers import Trainer

# Compute class weights from training set
label_list = train_tok["label"] if hasattr(train_tok["label"], "__iter__") else list(train_tok["label"])
label_array = np.array(label_list)

class_counts = np.bincount(label_array)
total = len(label_array)
class_weights = total / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

# Define WeightedTrainer
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights.to(self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

Class counts: [ 594  778 1092]
Class weights: tensor([1.3827, 1.0557, 0.7521])


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_lora",
    logging_dir="./logs_lora",

    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,


    learning_rate=3e-4,
    weight_decay=0.01,


    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    logging_steps=1,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
from transformers import Trainer

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.920482,0.572125,0.781174,0.782916,0.787013,0.781174
2,0.631475,0.541370,0.812958,0.817090,0.834352,0.812958
3,0.543676,0.532105,0.826406,0.825557,0.827678,0.826406
4,0.481499,0.545089,0.836186,0.836076,0.836995,0.836186
5,0.426827,0.548031,0.837408,0.837452,0.838687,0.837408


TrainOutput(global_step=1540, training_loss=0.6007917230779475, metrics={'train_runtime': 254.3359, 'train_samples_per_second': 48.44, 'train_steps_per_second': 6.055, 'total_flos': 818789581209600.0, 'train_loss': 0.6007917230779475, 'epoch': 5.0})

In [13]:
test_results = trainer.evaluate(eval_dataset=test_tok)

print("\n===== Test Set Results =====")
for key, value in test_results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")


===== Test Set Results =====
  eval_loss: 0.5637
  eval_accuracy: 0.8252
  eval_f1: 0.8249
  eval_precision: 0.8250
  eval_recall: 0.8252
  eval_runtime: 6.7849
  eval_samples_per_second: 120.5610
  eval_steps_per_second: 15.1810
  epoch: 5.0000


In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
trainer.save_model("/content/drive/MyDrive/csci316 project/lora_finetuned_xlmr")
tokenizer.save_pretrained("/content/drive/MyDrive/csci316 project/lora_finetuned_xlmr")
print("Model saved.")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model saved.
